# 03 · Finder baseline: "Show What's Trending"

The finder's job: out of all articles live on a given day, pick ~500 the user would probably like.

Before building the two-tower model, we need a number to beat. The simplest finder ignores who you are and shows everyone **what got the most clicks in the last 24 hours**. If the two-tower model can't beat this, it isn't worth its complexity.

We evaluate on the **validation day (Nov 14)** only.

In [1]:
import sys
sys.path.insert(0, "..")

import pandas as pd

from src.data import BUCKETS, explode_impressions, load_behaviors, load_news, split_by_time
from src.finder import hourly_clicks, rank, trending_scores, with_bucket_quotas
from src.metrics import evaluate

## The setup

- **Candidate pool:** every article shown to anyone on Nov 14 (and that has a bucket). This stands in for "what was live that day", since MIND doesn't give publish times.
- **What counts as a hit:** the articles the user clicked in that page view.

In [2]:
news = load_news()
behaviors = load_behaviors()
train, val = split_by_time(behaviors)

bucket_of = news.dropna(subset=["bucket"]).set_index("news_id")["bucket"].to_dict()
val_shown = explode_impressions(val)
pool = sorted(set(val_shown.news_id) & set(bucket_of))

# The clicked (bucketed) articles in each validation impression
val_clicks = (
    val_shown[(val_shown.clicked == 1) & val_shown.news_id.isin(bucket_of)]
    .groupby(["impression_id", "time"]).news_id.agg(list).reset_index()
)
print(f"Pool: {len(pool):,} articles  |  Validation impressions with a click: {len(val_clicks):,}")

Pool: 6,141 articles  |  Validation impressions with a click: 30,270


## How "trending" is measured

For each page view, count each article's clicks in the **24 hours before** it. That can include Nov 13 (train) and earlier on Nov 14, but never the current hour, so nothing peeks into the future.

To keep it fast we count clicks per hour, so every page view in the same hour gets the same trending list.

Here's what was trending at 9am on Nov 14:

In [3]:
hourly = hourly_clicks(behaviors)

example = trending_scores(hourly, pd.Timestamp("2019-11-14 09:00")).reindex(pool, fill_value=0).nlargest(10)
news.set_index("news_id").loc[example.index, ["bucket", "title"]].assign(clicks_last_24h=example.values)

,bucket,title,clicks_last_24h
news_id,,,
N287,News,Three school workers charged in death of speci...,1311
N4642,Entertain,Kodak Black Sentenced to Over 3 Years in Priso...,1187
N23446,Entertain,Prince Harry and Meghan Markle just shared a n...,1182
N56214,News,Deadly Arctic blast breaks records set more th...,1159
N38779,News,'One in a million' deer captured on camera in ...,843
N14029,Entertain,A master suite with an 'open-concept' bathroom...,823
N42457,Entertain,Meghan Markle and Prince Harry Won't Spend Chr...,678
N39010,News,Ruth Bader Ginsburg misses court due to illness,581
N47098,Entertain,Abandoned 'unicorn' puppy has a tail on its head,546


## Score it

We compare three finders, all returning 500 articles:

1. **Random:** pick 500 at random. This is the floor. The expected Recall@K is simply K ÷ pool size.
2. **Trending:** the top 500 by clicks in the last 24 hours.
3. **Trending + bucket quotas:** the top 100 from *each* of the 5 buckets. This is what the real finder must do so the arranger never runs out of any bucket.

**Metrics:**
- **Recall@K:** of the articles the user clicked, what share made it into the top K? Recall@500 is the main one: the finder's job is to not miss things.
- **NDCG@10:** are the clicked articles near the very top of the list? (1.0 = perfect)

In [4]:
rankings_trending, rankings_quota = [], []
for hour, group in val_clicks.groupby(val_clicks.time.dt.floor("h")):
    ranked = rank(trending_scores(hourly, hour), pool)
    quota = with_bucket_quotas(ranked, bucket_of, per_bucket=100)
    rankings_trending += [ranked] * len(group)
    rankings_quota += [quota] * len(group)

# groupby above walks impressions hour by hour, so line up the clicked lists in the same order
ordered_clicks = [c for _, g in val_clicks.groupby(val_clicks.time.dt.floor("h")) for c in g.news_id]

ks = (10, 50, 100, 500)
results = pd.DataFrame({
    "random": {**{f"recall@{k}": k / len(pool) for k in ks}, "ndcg@10": float("nan")},
    "trending": evaluate(rankings_trending, ordered_clicks, ks).mean(),
    "trending + quotas": evaluate(rankings_quota, ordered_clicks, ks).mean(),
}).T.round(3)
results

,recall@10,recall@50,recall@100,recall@500,ndcg@10
random,0.002,0.008,0.016,0.081,NaN
trending,0.095,0.350,0.485,0.786,0.048
trending + quotas,0.095,0.350,0.485,0.777,0.048


## Which buckets does trending miss?

Recall@500 split by the bucket of the clicked article. If trending only surfaces News and Entertain, the arranger could never fill a Learn slider. Quotas should fix that.

In [5]:
def recall_by_bucket(rankings, clicks, k=500):
    hits = [
        (bucket_of[news_id], news_id in set(ranked[:k]))
        for ranked, clicked in zip(rankings, clicks) for news_id in clicked
    ]
    return pd.DataFrame(hits, columns=["bucket", "hit"]).groupby("bucket").hit.mean()

by_bucket = pd.DataFrame({
    "trending": recall_by_bucket(rankings_trending, ordered_clicks),
    "trending + quotas": recall_by_bucket(rankings_quota, ordered_clicks),
    "articles in top 500 (trending, at midnight)": pd.Series([bucket_of[n] for n in rankings_trending[0][:500]]).value_counts(),
}).reindex(BUCKETS)
by_bucket.round(3)

,trending,trending + quotas,"articles in top 500 (trending, at midnight)"
News,0.739,0.694,116
Learn,0.768,0.799,86
Lifestyle,0.764,0.704,138
Entertain,0.852,0.853,93
Sports,0.602,0.691,67


## Takeaways

- **Trending is a strong baseline.** Its top 500 catches **79% of clicks** (random catches 8%). On MIND, a few hot stories get most of the clicks, so "what's hot right now" goes a long way. This is the number the two-tower model has to beat.
- **But it's bad at the top of the list.** NDCG@10 is only 0.05: everyone gets the same top 10, so it rarely puts *your* click first. This is where personalization should help most.
- **Quotas cost almost nothing.** Forcing 100 per bucket drops Recall@500 only from 0.786 to 0.777. It *raises* recall for Sports (0.60 → 0.69) and Learn (0.77 → 0.80), the buckets trending under-serves.
- **Sports is the hardest bucket for trending** (0.60 recall): sports fans click niche team stories that never make the global top list.

In [6]:
# Save the numbers so the two-tower notebook can compare against them
results.to_csv("../results/finder_baseline_metrics.csv")